# Assignment 11: Feature Engineering on the Titanic Dataset

Starting with a raw dataset: handle missing values, encode categorical variables, create at least three new features, remove irrelevant features, and explain how each transformation could improve model performance.

Dataset: **Titanic** (891 passengers). Target: `survived` (1 = survived, 0 = did not).

In [19]:
# STEP 1: Load the Raw Dataset

import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = sns.load_dataset('titanic')
print(f"Initial Dataset Shape: {df.shape}")
df.head()

Initial Dataset Shape: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [20]:
# STEP 2: Accuracy Before Feature Engineering

# Use only the raw numeric columns that have no missing values
base_features = ['pclass', 'sibsp', 'parch', 'fare']

Xb_train, Xb_test, yb_train, yb_test = train_test_split(df[base_features], df['survived'], test_size=0.2, random_state=42)

base_model = LogisticRegression(max_iter=1000)
base_model.fit(Xb_train, yb_train)

accuracy_before = accuracy_score(yb_test, base_model.predict(Xb_test))
print(f"Test Accuracy BEFORE feature engineering: {accuracy_before:.4f}")

Test Accuracy BEFORE feature engineering: 0.7151


In [21]:
# STEP 3: Handle Missing Values

print("Missing values before:")
print(df.isnull().sum())

# 'age': fill with the median age
df['age'] = df['age'].fillna(df['age'].median())

# 'embarked' and 'embark_town': fill with the most common value
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
df['embark_town'] = df['embark_town'].fillna(df['embark_town'].mode()[0])

# 'deck': 77% of the values are missing, too empty to fill, so drop it
df = df.drop(columns=['deck'])

print("\nMissing values after:", df.isnull().sum().sum())

Missing values before:
survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

Missing values after: 0


In [22]:
# STEP 4: Create New Features

# Total number of family members on board (including the passenger)
df['family_size'] = df['sibsp'] + df['parch'] + 1

# Ticket price per person (one ticket was often shared by a family)
df['fare_per_person'] = df['fare'] / df['family_size']

# Children (under 16) were given priority on lifeboats
df['is_child'] = (df['age'] < 16).astype(int)

df[['sibsp', 'parch', 'family_size', 'fare', 'fare_per_person', 'age', 'is_child']].head()

,sibsp,parch,family_size,fare,fare_per_person,age,is_child
0,1,0,2,7.2500,3.62500,22.0,0
1,1,0,2,71.2833,35.64165,38.0,0
2,0,0,1,7.9250,7.92500,26.0,0
3,1,0,2,53.1000,26.55000,35.0,0
4,0,0,1,8.0500,8.05000,35.0,0


In [23]:
# STEP 5: Remove Irrelevant Features

# 'alive'       -> same as the target 'survived' (would leak the answer)
# 'class'       -> duplicate of 'pclass'
# 'embark_town' -> duplicate of 'embarked'
# 'who', 'adult_male', 'alone' -> already covered by 'sex', 'age' and 'family_size'
df = df.drop(columns=['alive', 'class', 'embark_town', 'who', 'adult_male', 'alone'])

print(f"Dataset Shape after removing irrelevant features: {df.shape}")
df.head()

Dataset Shape after removing irrelevant features: (891, 11)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,family_size,fare_per_person,is_child
0,0,3,male,22.0,1,0,7.2500,S,2,3.62500,0
1,1,1,female,38.0,1,0,71.2833,C,2,35.64165,0
2,1,3,female,26.0,0,0,7.9250,S,1,7.92500,0
3,1,1,female,35.0,1,0,53.1000,S,2,26.55000,0
4,0,3,male,35.0,0,0,8.0500,S,1,8.05000,0


In [24]:
# STEP 6: Define Feature Groups

X = df.drop('survived', axis=1)
y = df['survived']

numerical_features = ['age', 'sibsp', 'parch', 'fare', 'family_size', 'fare_per_person', 'is_child', 'pclass']
nominal_features = ['sex', 'embarked']

In [25]:
# STEP 7: Encode Categorical Variables (and Scale Numerical Ones)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('nom', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_features)
    ],
    remainder='drop'
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit on the training data, then transform both train and test sets
X_train_clean = preprocessor.fit_transform(X_train)
X_test_clean = preprocessor.transform(X_test)

print(f"Cleaned Training Data Shape: {X_train_clean.shape}")
print("Columns:", list(preprocessor.get_feature_names_out()))

Cleaned Training Data Shape: (712, 13)
Columns: ['num__age', 'num__sibsp', 'num__parch', 'num__fare', 'num__family_size', 'num__fare_per_person', 'num__is_child', 'num__pclass', 'nom__sex_female', 'nom__sex_male', 'nom__embarked_C', 'nom__embarked_Q', 'nom__embarked_S']


In [26]:
# STEP 8: Accuracy After Feature Engineering

model = LogisticRegression(max_iter=1000)
model.fit(X_train_clean, y_train)

accuracy_after = accuracy_score(y_test, model.predict(X_test_clean))
print(f"Test Accuracy BEFORE feature engineering: {accuracy_before:.4f}")
print(f"Test Accuracy AFTER feature engineering:  {accuracy_after:.4f}")


Test Accuracy BEFORE feature engineering: 0.7151
Test Accuracy AFTER feature engineering:  0.7989


## How Each Transformation Could Improve Model Performance

Feature engineering raised the test accuracy from **71.5%** to **79.9%** with the same model and the same test passengers.

- **Filling missing values:** models can't train on empty values. Filling `age` and `embarked` keeps all 891 passengers instead of throwing rows away, so the model learns from more data.
- **Dropping `deck`:** with 77% of values missing, filling it would mostly be guessing and add noise.
- **Encoding `sex` and `embarked`:** models only understand numbers. One-hot encoding turns the text into 0/1 columns without implying any order. `sex` is the strongest predictor of survival, so the model needs it.
- **Scaling:** puts `fare` (up to 500) and `age` (up to 80) on the same scale, so no feature dominates just because of its units.
- **`family_size`:** combines `sibsp` and `parch` into one clearer signal. Small families survived more than people travelling alone or in large families.
- **`fare_per_person`:** a fairer measure of wealth than the raw ticket price, and wealthier passengers survived more.
- **`is_child`:** captures the "women and children first" rule directly.
- **Removing irrelevant features:** `alive` is the answer itself, so keeping it would give a fake perfect score. Duplicate columns add no new information, only noise, which can lead to overfitting.